## CineBot — a movie ticket booking agent


Covers: models, messages, tools (dummy data, booking, cancellation, showtimes), 
and dynamic tool selection based on customer tier (VIP vs normal).

In [1]:
%pip install -qU langchain langchain-openai langgraph pydantic
%pip install -qU rich

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from rich.console import Console
from rich.panel import Panel
from rich.pretty import pprint

console = Console()

def pretty_print_messages(result: dict) -> None:
    """Render each message in an agent result with role-colored panels."""
    styles = {"human": "cyan", "ai": "green", "tool": "yellow", "system": "magenta"}
    for message in result["messages"]:
        role = message.type
        console.print(Panel(str(message.content), title=role.upper(), border_style=styles.get(role, "white")))

#### 1. Configure API key

In [3]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    api_key = os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")
    print(f"OpenRouter API key set in environment variable OPENROUTER_API_KEY: {api_key[0:4]}")

OpenRouter API key set in environment variable OPENROUTER_API_KEY: sk-o


In [4]:
from langchain_openai import ChatOpenAI

# OpenRouter exposes an OpenAI-compatible API, so ChatOpenAI works with a custom base_url
model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.2'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000021A5EAA7F80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021A6084FE30>, root_client=<openai.OpenAI object at 0x0000021A5CAF85F0>, root_async_client=<openai.AsyncOpenAI object at 0x0000021A60608B90>, model_name='openai/gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', openai_proxy=None, stream_chunk_timeout=120.0)

#### 2. Dummy cinema data
In-memory data standing in for a real database: movies, showtimes, seats, and bookings.

In [5]:
from datetime import datetime

# demo data store, resets whenever the kernel restarts
CINEMAS = {
    "PVR-Downtown": {
        "movies": {
            "Interstellar 2": ["14:00", "18:00", "21:30"],
            "The Great Heist": ["15:30", "19:00"],
        }
    },
    "INOX-Mall": {
        "movies": {
            "Interstellar 2": ["13:00", "17:00"],
            "Comedy Nights": ["16:00", "20:00"],
        }
    },
}

# seat_id -> booking record
BOOKINGS: dict[str, dict] = {}
_next_booking_id = 1

def _new_booking_id() -> str:
    global _next_booking_id
    booking_id = f"BK-{_next_booking_id:04d}"
    _next_booking_id += 1
    return booking_id

#### 3. Messages
Build a simple conversation using LangChain message types before wiring up tools.

In [6]:
from langchain.messages import SystemMessage, HumanMessage

system_prompt = SystemMessage(content=(
    "You are CineBot, a movie ticket booking assistant. "
    "Use the available tools to check showtimes, book, and cancel tickets."
))
user_message = HumanMessage(content="What movies are playing at PVR-Downtown today?")

response = model.invoke([system_prompt, user_message])
print(response.content)

I currently don't have access to real-time data to check the showtimes for PVR-Downtown. However, you can easily find the latest showtimes by visiting the PVR Cinemas website or using their mobile app. If you need help with anything else, feel free to ask!


#### 4. Tools — showtimes, booking, and cancellation
Each tool operates on the dummy `CINEMAS`/`BOOKINGS` data defined above.

In [7]:
from langchain.tools import tool

@tool
def get_showtimes(cinema:str,movie:str)-> dict:
    """Get showtimes for a movie at a cinema."""
    cinema_data = CINEMAS.get(cinema)
    if not cinema_data:
        return {"error": f"Cinema '{cinema}' not found."}
    times = cinema_data["movies"].get(movie)
    if times is None:
        return {"error": f"Movie '{movie}' not found at cinema '{cinema}'."}
    return {"found": True, "cinema": cinema, "movie": movie, "showtimes": times}


get_showtimes.invoke({"cinema": "PVR-Downtown", "movie": "Interstellar 2"})

{'found': True,
 'cinema': 'PVR-Downtown',
 'movie': 'Interstellar 2',
 'showtimes': ['14:00', '18:00', '21:30']}

In [8]:
@tool("ticket_booking",description="Book tickets for a movie at a cinema and showtime.")
def book_ticket(cinema: str, movie: str, showtime: str, seats: int) -> dict:
    """Book tickets for a movie at a cinema and showtime. Returns a booking confirmation."""
    cinema_data = CINEMAS.get(cinema)
    if cinema_data is None or showtime not in cinema_data["movies"].get(movie, []):
        return {"success": False, "message": f"{movie} at {showtime} not found at {cinema}"}

    booking_id = _new_booking_id()
    BOOKINGS[booking_id] = {
        "cinema": cinema,
        "movie": movie,
        "showtime": showtime,
        "seats": seats,
        "status": "confirmed",
    }
    return {"success": True, "booking_id": booking_id, **BOOKINGS[booking_id]}

book_ticket.invoke({"cinema": "PVR-Downtown", "movie": "Interstellar 2", "showtime": "18:00", "seats": 2})

{'success': True,
 'booking_id': 'BK-0001',
 'cinema': 'PVR-Downtown',
 'movie': 'Interstellar 2',
 'showtime': '18:00',
 'seats': 2,
 'status': 'confirmed'}

In [9]:
@tool
def cancel_booking(booking_id: str) -> dict:
    """Cancel an existing booking by its booking ID."""
    booking = BOOKINGS.get(booking_id)
    if booking is None:
        return {"success": False, "message": f"No booking found for {booking_id}"}
    if booking["status"] == "cancelled":
        return {"success": False, "message": f"Booking {booking_id} is already cancelled"}

    booking["status"] = "cancelled"
    return {"success": True, "booking_id": booking_id, "message": "Booking cancelled"}

cancel_booking.invoke({"booking_id": "BK-0001"})

{'success': True, 'booking_id': 'BK-0001', 'message': 'Booking cancelled'}

In [10]:
@tool
def get_booking_status(booking_id: str) -> dict:
    """Get the status of an existing booking by its booking ID."""
    booking = BOOKINGS.get(booking_id)
    if booking is None:
        return {"success": False, "message": f"No booking found for {booking_id}"}
    return {"success": True, "booking_id": booking_id, "status": booking["status"], **booking}

In [11]:
@tool(return_direct=True)
def refund_policy() -> str:
    """Return the cinema's refund policy. Use when the user asks about refunds or cancellation rules."""
    return (
        "Refund Policy: Tickets cancelled at least 2 hours before showtime are fully refunded. "
        "Cancellations within 2 hours of showtime are non-refundable."
    )

refund_policy.invoke({})

'Refund Policy: Tickets cancelled at least 2 hours before showtime are fully refunded. Cancellations within 2 hours of showtime are non-refundable.'

#### 5. Short-term memory — InMemorySaver with thread_id
A checkpointer persists conversation state per `thread_id`, so the agent remembers earlier turns within the same thread.

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

checkpointer = InMemorySaver()

agent = create_agent(
    model,
    tools=[get_showtimes, book_ticket, cancel_booking,get_booking_status, refund_policy],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "customer-101"}}
result = agent.invoke({"messages": [HumanMessage(content="What showtimes are available for Interstellar 2 at PVR-Downtown?")]}, config)
print(result["messages"][-1].content)

The available showtimes for "Interstellar 2" at PVR-Downtown are:

- 14:00
- 18:00
- 21:30


In [13]:
result = agent.invoke({"messages": [HumanMessage(content="Book 2 tickets for Interstellar 2 at 18:00 at PVR-Downtown.")]}, config)
print(result["messages"][-1].content)

Your booking for 2 tickets to "Interstellar 2" at 18:00 at PVR-Downtown has been confirmed. 

**Booking ID:** BK-0002  
**Status:** Confirmed


In [14]:
# reuses the same thread_id, so the agent has access to the earlier turn
result = agent.invoke({"messages": [HumanMessage(content="How many tickets are booked for Interstellar 2 at PVR-Downtown?")]}, config)
print(result["messages"][-1].content)

You have booked a total of **2 tickets** for "Interstellar 2" at PVR-Downtown for the showtime of 18:00.


In [15]:
####### For new customer 102    

config = {"configurable": {"thread_id": "customer-102"}}
result = agent.invoke({"messages": [HumanMessage(content="How many tickets are booked for Interstellar 2 at PVR-Downtown? for customer-102")]}, config)
print(result["messages"][-1].content)

It appears that there are no bookings found for "Interstellar 2" at PVR-Downtown under the ID "customer-102." If you need assistance with anything else, please let me know!


#### 7. Dynamic tool selection — VIP vs normal customers
Choose the tool list per request based on customer tier, then bind only those tools to the model.

In [16]:
# customer_id -> tier, the source of truth for access control (not user-supplied)
CUSTOMERS = {
    "customer-101": "normal",
    "customer-102": "normal",
    "vip-customer-1": "vip",
}

from langchain.tools import ToolRuntime

@tool
def priority_booking(cinema: str, movie: str, showtime: str, seats: int, runtime: ToolRuntime) -> dict:
    """VIP-only: book best available seats with priority handling and no seat limit."""
    customer_id = runtime.context.get("customer_id") if runtime.context else None
    if CUSTOMERS.get(customer_id) != "vip":
        return {"success": False, "message": "Priority booking is available to VIP customers only."}

    cinema_data = CINEMAS.get(cinema)
    if cinema_data is None or showtime not in cinema_data["movies"].get(movie, []):
        return {"success": False, "message": f"{movie} at {showtime} not found at {cinema}"}

    booking_id = _new_booking_id()
    BOOKINGS[booking_id] = {
        "cinema": cinema, "movie": movie, "showtime": showtime,
        "seats": seats, "status": "confirmed", "priority": True,
    }
    return {"success": True, "booking_id": booking_id, **BOOKINGS[booking_id]}

NORMAL_TOOLS = [get_showtimes, book_ticket, cancel_booking, get_booking_status, refund_policy]
VIP_TOOLS = [get_showtimes, priority_booking, book_ticket, cancel_booking, get_booking_status, refund_policy]

def get_agent_for_customer(customer_id: str):
    """Look up the real tier for this customer_id, then build an agent scoped to it."""
    tier = CUSTOMERS.get(customer_id, "normal")
    tools = VIP_TOOLS if tier == "vip" else NORMAL_TOOLS
    return create_agent(model, tools=tools, checkpointer=checkpointer, context_schema=dict)

In [17]:
vip_agent = get_agent_for_customer("vip-customer-1")
vip_config = {"configurable": {"thread_id": "vip-customer-1"}}
result = vip_agent.invoke(
    {"messages": [HumanMessage(content="Book 3 priority seats for Interstellar 2 at 21:30 at PVR-Downtown.")]},
    vip_config,
    context={"customer_id": "vip-customer-1"},
)
print(result["messages"][-1].content)

c:\Users\visha\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value={'customer_id': 'vip-customer-1'}, input_type=dict])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\visha\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value={'customer_id': 'vip-customer-1'}, input_type=dict])
  return self.__pydantic_serializer__.to_python(


Your priority booking for 3 seats for "Interstellar 2" at PVR-Downtown at 21:30 has been confirmed. 

- **Booking ID:** BK-0003
- **Status:** Confirmed

Enjoy the movie!


In [18]:
normal_agent = get_agent_for_customer("customer-102")
# normal_config = {"configurable": {"thread_id": "customer-102"}}
result = normal_agent.invoke(
    {"messages": [HumanMessage(content="Book 3 priority seats for Interstellar 2 at 21:30 at PVR-Downtown.")]},
    config,
    context={"customer_id": "customer-102"},
)
print(result["messages"][-1].content)

Your booking for 3 priority seats for "Interstellar 2" at PVR-Downtown at 21:30 has been confirmed. Your booking ID is **BK-0004**. If you need any further assistance, feel free to ask!


In [22]:
# normal_agent = get_agent_for_customer("customer-102")
# normal_config = {"configurable": {"thread_id": "customer-102"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="How many bookings done for customer-102?")]},
    config,
    context={"customer_id": "customer-102"},
)
print(result["messages"][-1].content)

There are no bookings found for "customer-102" either. If you have any other questions or need assistance, please let me know!
